# 0. Reestruturação da planilha bruta — Thermal Predictor

O arquivo `data/dataset_bruto.csv` (exportado da aba **"Option 1 - Last Cases"**, com correções manuais de formatação) está em formato "largo": cada **Run** é uma coluna e cada **termopar (TC)** é uma linha, com as specs de cada run (Reason, Blower Direction, Housing, etc.) empilhadas em linhas de cabeçalho acima da tabela de temperaturas.

Este notebook faz **apenas a reestruturação do formato** (wide → long): nenhum dado é calculado, filtrado ou alterado — só redistribuído. O resultado é um dataframe com uma linha por combinação (Run × TC), contendo as specs do run e do TC como colunas, salvo em `data/dataset_pre_processado.csv`.

### Por que reestruturar?

No formato "largo" (wide), cada Run é uma coluna e cada TC é uma linha — ótimo para leitura humana em planilha, mas ruim para análise: para responder algo como "qual foi a temperatura do TC 5 no Run 12?" seria preciso navegar linha/coluna manualmente, e um modelo preditivo não consegue usar a tabela assim, pois espera **uma observação por linha** (aqui, uma combinação Run × TC), com todas as variáveis relevantes (specs do run, specs do TC, temperatura medida) como colunas dessa mesma linha. Essa é a forma "longa" (long format), e é o que este notebook produz.

### Glossário rápido

- **Run**: um experimento/teste específico, com uma combinação de condições (direção do soprador, tipo de housing, etc.).
- **TC (termopar)**: sensor que mede a temperatura em um ponto específico do fogão.
- **Key Area**: região do fogão onde aquele TC está localizado (ex: Mainback, Coil Temps).
- **Active Limit**: temperatura limite considerada segura/aceitável para aquele ponto.
- **Blower Direction / Housing / Board Position**: parâmetros de configuração física do teste (para onde o soprador aponta, se a carcaça está aberta ou fechada, etc.).

**Entrada:** `data/dataset_bruto.csv`, a planilha bruta do parceiro.

**Saída:** `data/dataset_pre_processado.csv` e `data/classificacao_colunas.csv`, lidas pelos notebooks seguintes.

Os arquivos de `data/` não estão no repositório (dados confidenciais do parceiro);
veja a seção "Os notebooks" do README para obtê-los e para a ordem de execução.

## 1. Carregar a planilha bruta

In [ ]:
import pandas as pd

raw = pd.read_csv('../data/dataset_bruto.csv', header=None)
raw.shape

## 2. Specs por Run

As linhas 0–8 (colunas 4 em diante) trazem, uma spec por linha, os metadados de cada run: rótulo, Reason, Blower Direction, Housing, Board Position, Run Case, Losses, Deflector Hole e Material Coil Support.

Esses índices (linhas 0–8, colunas a partir da 4) não são arbitrários: foram lidos direto da planilha bruta olhando o CSV aberto (`raw.head(10)` mostra as primeiras linhas e confirma onde cada spec está). As colunas 0–3 são reservadas para as specs de TC (seção 3) e só a partir da coluna 4 começam os dados de cada run.

Extraímos esse bloco com `raw.loc[...]` e aplicamos `.T` (transpose) para **girar a tabela**: na planilha bruta cada spec é uma linha e cada run é uma coluna; depois do `.T`, cada run passa a ser uma linha e cada spec uma coluna — formato mais fácil de trabalhar com um dataframe (uma linha = um run).

In [ ]:
RUN_SPEC_ROWS = {
    0: 'run_label',
    1: 'reason',
    2: 'blower_direction',
    3: 'housing',
    4: 'board_position',
    5: 'run_case',
    6: 'losses',
    7: 'deflector_hole',
    8: 'material_coil_support',
}
RUN_COL_START = 4

run_block = raw.loc[list(RUN_SPEC_ROWS.keys()), RUN_COL_START:]
run_meta = run_block.T
run_meta.columns = [RUN_SPEC_ROWS[i] for i in run_block.index]
run_meta.index.name = 'col_pos'
run_meta = run_meta.reset_index()
run_meta.head()

### 2.1 Desambiguar rótulos de run duplicados

A planilha bruta reusa o mesmo rótulo de run em algumas colunas (ex: "Run 22" aparece 2x). Para gerar um `run_id` único por coluna, mantemos o rótulo original na primeira ocorrência e sufixamos as repetições: `"Run 22"`, `"Run 22 (2)"`.

In [ ]:
dup_count = run_meta.groupby('run_label').cumcount()
run_meta['run_id'] = [
    label if dup_count[i] == 0 else f"{label} ({dup_count[i] + 1})"
    for i, label in enumerate(run_meta['run_label'])
]

print(f"{run_meta['run_id'].nunique()} run_id únicos de {len(run_meta)} runs")
run_meta[run_meta['run_label'].duplicated(keep=False)][['col_pos', 'run_label', 'run_id']]

## 3. Specs por TC (termopar)

A partir da linha 9 (a linha 0–8 é o bloco de specs de run da seção 2 acima), colunas 0–3 trazem `TC Number`, `TC Name`, `Key Area` e `Active Limit` para cada termopar. Intercaladas nessas linhas existem **linhas de cabeçalho de seção** (ex: "Mainback Components", "IPC Components") — são linhas que, na planilha bruta, servem só para separar visualmente os TCs em grupos por região do fogão, e por isso têm quase todas as colunas vazias.

Para detectá-las automaticamente usamos a regra "linha com no máximo 2 valores preenchidos": uma linha de TC normal tem as 4 colunas de spec preenchidas (ou pelo menos 3), enquanto uma linha de cabeçalho de seção só tem o nome da seção na coluna 0 e o resto vazio. Esse limite (`<= 2`) dá uma margem de tolerância para pequenas inconsistências de formatação sem confundir as duas coisas.

Essas linhas de cabeçalho servem apenas para preencher (`ffill`) os poucos `Key Area` vazios dentro de cada bloco — a coluna `Key Area` já vem preenchida na maior parte das linhas.

In [ ]:
section_rows = [
    i for i in range(9, raw.shape[0])
    if raw.iloc[i].notna().sum() <= 2 and pd.notna(raw.iloc[i, 0])
]
print("Linhas de cabeçalho de seção:", section_rows)
list(raw.iloc[section_rows, 0])

In [ ]:
tc_rows = [i for i in range(9, raw.shape[0]) if i not in section_rows]

tc_meta = raw.loc[tc_rows, [0, 1, 2, 3]].copy()
tc_meta.columns = ['tc_number', 'tc_name', 'key_area', 'active_limit']

bounds = section_rows + [raw.shape[0]]
for b in range(len(section_rows)):
    start, end = section_rows[b] + 1, bounds[b + 1]
    block_idx = [i for i in tc_meta.index if start <= i < end]
    tc_meta.loc[block_idx, 'key_area'] = tc_meta.loc[block_idx, 'key_area'].ffill()

print("TCs:", tc_meta.shape[0], "| Key Area vazio após ffill:", tc_meta['key_area'].isna().sum())
tc_meta.head()

## 4. Melt: de largo (Run × TC em matriz) para longo (uma linha por Run × TC)

Esta é a transformação central do notebook. Hoje `temps` é uma matriz: cada linha é um TC, cada coluna é um run, e cada célula é a temperatura medida daquele TC naquele run — o formato "largo" descrito na introdução.

`pd.melt()` "derrete" essa matriz em uma tabela longa: pega cada célula da matriz e transforma em uma linha própria, com uma coluna dizendo de qual run ela veio (`var_name='run_id'`) e outra com o valor da temperatura (`value_name='temp'`). As colunas passadas em `id_vars` (as specs do TC) são repetidas em cada uma dessas novas linhas, porque continuam valendo para aquele TC independente do run.

Depois do melt, cada linha do resultado já identifica o TC (via `id_vars`) e o run (via `run_id`), mas ainda faltam as specs *do run* (Reason, Housing, etc.) — por isso, na sequência, fazemos um `merge` com `run_meta` para trazer essas colunas também, usando `run_id` como chave de junção.

In [ ]:
temps = raw.loc[tc_rows, RUN_COL_START:].copy()
temps.columns = run_meta['run_id'].values
temps.index = tc_meta.index

long_df = temps.join(tc_meta).melt(
    id_vars=['tc_number', 'tc_name', 'key_area', 'active_limit'],
    var_name='run_id',
    value_name='temp',
)

run_spec_cols = ['run_id', 'reason', 'blower_direction', 'housing', 'board_position',
                 'run_case', 'losses', 'deflector_hole', 'material_coil_support']
long_df = long_df.merge(run_meta[run_spec_cols], on='run_id', how='left')

# uma célula em branco na planilha original = TC não medido nesse run (não é dado a reestruturar)
long_df = long_df.dropna(subset=['temp']).reset_index(drop=True)

# a planilha bruta usa vírgula como separador decimal (formato pt-BR)
long_df['temp'] = pd.to_numeric(long_df['temp'].str.replace(',', '.', regex=False))

long_df = long_df[[
    'run_id', 'tc_number', 'tc_name', 'key_area', 'active_limit', 'temp',
    'reason', 'blower_direction', 'housing', 'board_position',
    'run_case', 'losses', 'deflector_hole', 'material_coil_support',
]]
long_df.shape

## 5. Conferir o resultado

In [ ]:
long_df.info()

In [ ]:
long_df.shape

In [ ]:
long_df.head(20)

In [ ]:
# runs presentes e conferência de que os duplicados foram desambiguados
print(long_df['run_id'].nunique(), "runs distintos")
assert long_df.groupby('run_id').ngroups == run_meta['run_id'].nunique()
sorted(long_df['run_id'].unique())

## 6. Salvar em `data/dataset_pre_processado.csv`

In [ ]:
from pathlib import Path

caminho_dataset = Path('../data/dataset_pre_processado.csv')

long_df.to_csv(caminho_dataset, index=False, encoding='utf-8-sig')
print(f"Salvo em {caminho_dataset.resolve()}: {long_df.shape}")

# 1. Exploração de Dados — Thermal Predictor

## 1.1 Classificação das colunas — Numérica vs. Categórica

Insumo para a Seção 4.2.1 da documentação (Exploração de dados).

Fonte: `data/dataset_pre_processado.csv`, gerado pela seção [`0. Reestruturação da planilha bruta`](#0-reestruturação-da-planilha-bruta-—-thermal-predictor) acima — já está no formato "tidy" (uma linha por combinação Run x TC), então esta seção **não altera dado nenhum**, só lê e classifica. Por isso fica separada das seções de Exploração/Pré-processamento que manipulam o dado real.

Metodologia: `df.dtypes` como ponto de partida, seguido de revisão manual coluna a coluna para pegar casos ambíguos (ex.: coluna que parece numérica mas é identificador, ou tem valores textuais misturados).


### 1.1.1 Carregar o dataset tidy

In [ ]:
import pandas as pd

df = pd.read_csv('../data/dataset_pre_processado.csv')
df.shape

### 1.1.2 Baseline com `df.dtypes`

Ponto de partida obrigatório pela task — mas o pandas infere tipo pela sintaxe do valor, não pelo significado da coluna, então precisa de revisão manual em seguida.

In [ ]:
df.dtypes

### 1.1.3 Revisão manual das colunas que parecem numéricas

`df.dtypes` mostra tudo como `str`/`object` porque o CSV tem sujeira de formatação (identificadores misturados com número, texto onde deveria ter número, etc.). Antes de classificar, checamos com `pd.to_numeric(..., errors='coerce')` quais colunas *realmente* convertem para número de forma limpa, e o que sobra quando não convertem — isso é o que decide se a coluna é Numérica de fato ou só parece.

In [ ]:
candidatas_numericas = ['tc_number', 'active_limit', 'temp']

for col in candidatas_numericas:
    coerced = pd.to_numeric(df[col], errors='coerce')
    nao_numericos = df[col][coerced.isna() & df[col].notna()].unique()
    print(f"{col}: {len(nao_numericos)} valores nao numericos -> {list(nao_numericos)[:10]}")

**Leitura dos 3 casos:**

- `tc_number`: já converte 100% para número (o `#REF!` que existia numa versão anterior do dataset foi corrigido na reestruturação). Mesmo assim, continua sendo um **identificador** do sensor, não uma quantidade — não faz sentido calcular média/desvio de `tc_number`. → **Categórica**.
- `active_limit`: ainda resta 1 valor não numérico (`"Monitor (150 C)"`) — sensor sem limite de segurança definido (só monitorado, sem threshold). O núcleo da coluna é uma temperatura limite em °C. → **Numérica**, mas esse valor residual precisa virar `NaN` (ou um flag `is_monitor` à parte) no pré-processamento (Seção 4.2.2) antes de qualquer cálculo estatístico.
- `temp`: já converte 100% para número (as strings formatadas como porcentagem que existiam numa versão anterior do dataset foram corrigidas na reestruturação). → **Numérica**, sem ressalvas.

As demais colunas (`run_id`, `tc_name`, `key_area`, `reason`, `blower_direction`, `housing`, `board_position`, `run_case`, `losses`, `deflector_hole`, `material_coil_support`) são texto/rótulo por natureza — sem ambiguidade, todas **Categóricas**.

### 1.1.4 Tabela final de classificação

In [ ]:
# colunas que sao de fato Numericas depois da revisao manual (secao 3);
# tudo que nao estiver aqui e Categorica, mesmo que df.dtypes tenha mostrado
# algo parecido com numero (caso do tc_number)
NUMERICAS = {'active_limit', 'temp'}

rows = []
for col in df.columns:
    tipo = 'Numerica' if col in NUMERICAS else 'Categorica'
    rows.append({
        'coluna': col,
        'tipo': tipo,
        'n_unicos': int(df[col].nunique()),
        'ausentes': int(df[col].isna().sum()),
    })

classificacao = pd.DataFrame(rows)
classificacao

In [ ]:
classificacao.to_csv('../data/classificacao_colunas.csv', index=False)
print('salvo em data/classificacao_colunas.csv')

### 1.1.5 Markdown pronto para a Seção 4.2.1

In [ ]:
def to_markdown(d, cols):
    header = '| ' + ' | '.join(cols) + ' |'
    sep = '|' + '|'.join(['---'] * len(cols)) + '|'
    linhas = [header, sep]
    for _, row in d.iterrows():
        linhas.append('| ' + ' | '.join(str(row[c]) for c in cols) + ' |')
    return '\n'.join(linhas)

print(to_markdown(classificacao, ['coluna', 'tipo']))

Estatistica descritiva das colunas: 

## 1.2 Estatística descritiva das colunas

A partir da classificação de colunas feita na seção [`1.1 Classificação das colunas`](#1.1-Classificação-das-colunas-—-Numérica-vs.-Categórica) acima
(numéricas: `temp`, `active_limit`; categóricas: as demais 12 colunas), calculei aqui as
métricas descritivas de cada uma: média, desvio-padrão e quartis para as numéricas; valores
únicos, moda e frequência para as categóricas.

Fonte: `data/dataset_pre_processado.csv` (gerado pela seção
[`0. Reestruturação da planilha bruta`](#0-reestruturação-da-planilha-bruta-—-thermal-predictor) acima).

**Escopo desta seção:** apenas cálculo das métricas e montagem da tabela, nenhum dado é
tratado, filtrado ou interpretado aqui. Inconsistências já identificadas na classificação
(ex.: o valor residual de texto em `active_limit`, as grafias diferentes de
`blower_direction`) são reportadas como estão; o tratamento delas fica para a Seção 4.2.2
(pré-processamento).

In [ ]:
import pandas as pd

CAMINHO = "../data/dataset_pre_processado.csv"

df = pd.read_csv(CAMINHO)
print(f"Dataset: {df.shape[0]} linhas, {df.shape[1]} colunas\n")

COLUNAS_NUMERICAS = ["temp", "active_limit"]
COLUNAS_CATEGORICAS = ["run_id", "tc_number", "tc_name", "key_area", "reason",
                        "blower_direction", "housing", "board_position",
                        "run_case", "losses", "deflector_hole",
                        "material_coil_support"]


# =============================================================================
# 1. ESTATÍSTICA DESCRITIVA - COLUNAS NUMÉRICAS
# =============================================================================
def estatisticas_numericas(dataframe, colunas):
    """
    Média, mediana, desvio-padrão, min, max, Q1 e Q3 de cada coluna numérica.
    `active_limit` é convertida com pd.to_numeric(errors='coerce'): o único
    valor não numérico da coluna ("Monitor (150 C)", 56 ocorrências, já
    documentado na classificação do Card 1.1) vira NaN só para permitir o
    cálculo — nenhum tratamento além disso é feito aqui.
    """
    df_numerico = dataframe[colunas].apply(pd.to_numeric, errors="coerce")
    return df_numerico.describe(percentiles=[.25, .5, .75]).round(2)


print("=" * 60)
print("1. ESTATISTICA DESCRITIVA - COLUNAS NUMERICAS")
print("=" * 60)
tab_num = estatisticas_numericas(df, COLUNAS_NUMERICAS)
print(tab_num.to_string())

# =============================================================================
# 2. ESTATÍSTICA DESCRITIVA - COLUNAS CATEGÓRICAS
# =============================================================================
def estatisticas_categoricas(dataframe, colunas):
    """
    Moda, número de entradas, valores nulos e valores únicos de cada
    coluna categórica. Retorna um resumo consolidado (uma linha por coluna).
    """
    linhas = []
    for col in colunas:
        s = dataframe[col]
        linhas.append({
            "coluna": col,
            "moda": s.mode()[0] if not s.mode().empty else None,
            "entradas": s.count(),
            "nulos": s.isna().sum(),
            "unicos": s.nunique(),
        })
    return pd.DataFrame(linhas)


print("\n" + "=" * 60)
print("2. ESTATISTICA DESCRITIVA - COLUNAS CATEGORICAS (resumo)")
print("=" * 60)
tab_cat = estatisticas_categoricas(df, COLUNAS_CATEGORICAS)
print(tab_cat.to_string(index=False))

# Frequência completa (value_counts) das categóricas de baixa/média
# cardinalidade — para run_id, tc_number e tc_name (56-77 valores únicos
# cada) o resumo acima já cobre o pedido; a distribuição completa deles
# fica extensa demais para caber em tabela e não agrega além do resumo.
COLUNAS_PARA_FREQUENCIA = ["key_area", "reason", "blower_direction", "housing",
                            "board_position", "run_case", "losses",
                            "deflector_hole", "material_coil_support"]

print("\n" + "=" * 60)
print("FREQUÊNCIA (value_counts) POR COLUNA")
print("=" * 60)
for col in COLUNAS_PARA_FREQUENCIA:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).to_string())

## 1.3 Gráficos exploratórios — relação entre variáveis

Insumo para a Seção 4.2.1 da documentação (Exploração de dados). Objetivo: visualizar como `temp` se
relaciona com variáveis de configuração e de localização do ensaio (`key_area`, `housing`,
`blower_direction`, `material_coil_support`) e com o próprio `active_limit`.

Fonte: `data/dataset_pre_processado.csv`. Nenhum dado é alterado aqui, só lido e plotado; o placeholder
`active_limit = 1000` ("sem limite definido", ver Seção 1.1) é excluído apenas do gráfico
`temp` vs. `active_limit`, para não distorcer a escala do eixo.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

df_expl = pd.read_csv('../data/dataset_pre_processado.csv')
df_expl.columns = df_expl.columns.str.strip()
# 'active_limit' tem 1 valor textual ("Monitor (150 C)", Secao 1.1) que forca a coluna
# inteira para string; convertida aqui so para uso nos graficos, sem alterar o CSV.
df_expl['active_limit'] = pd.to_numeric(df_expl['active_limit'], errors='coerce')
# 'blower_direction' ainda tem grafias duplicadas nesta etapa (Secao 1, antes da
# padronizacao da Secao 2 - item 5.1); normalizada aqui so para o eixo do grafico.
df_expl['blower_direction'] = df_expl['blower_direction'].str.strip().str.title()

FIG_DIR = '../assets/graficos'
os.makedirs(FIG_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# Figura 11: temp por key_area
# ---------------------------------------------------------------------------
ordem_area = df_expl.groupby('key_area')['temp'].median().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(9, 5.5))
dados = [df_expl.loc[df_expl['key_area'] == a, 'temp'].dropna() for a in ordem_area]
bp = ax.boxplot(dados, tick_labels=ordem_area, patch_artist=True, showfliers=True)
for box in bp['boxes']:
    box.set_facecolor('#4C72B0')
    box.set_alpha(0.75)
ax.set_ylabel('Temperatura (\u00b0C)')
ax.set_xlabel('\u00c1rea do produto (key_area)')
ax.set_title('Temperatura por \u00e1rea do produto (key_area)', fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/temp_por_area.png', bbox_inches='tight')
plt.show()

# ---------------------------------------------------------------------------
# Figura 12: temp por housing x blower_direction
# ---------------------------------------------------------------------------
pivot = (df_expl.groupby(['housing', 'blower_direction'])['temp']
         .mean().reset_index()
         .pivot(index='housing', columns='blower_direction', values='temp'))

fig, ax = plt.subplots(figsize=(9, 5.5))
pivot.plot(kind='bar', ax=ax, color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'])
ax.set_ylabel('Temperatura m\u00e9dia (\u00b0C)')
ax.set_xlabel('Configura\u00e7\u00e3o da carca\u00e7a (housing)')
ax.set_title('Temperatura m\u00e9dia por housing e dire\u00e7\u00e3o do blower', fontweight='bold')
ax.legend(title='blower_direction', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/temp_housing_blower.png', bbox_inches='tight')
plt.show()

print(pivot.round(2))

# ---------------------------------------------------------------------------
# Figura 13: taxa de excedencia do active_limit por material_coil_support
# ---------------------------------------------------------------------------
tmp = df_expl.dropna(subset=['active_limit']).copy()
tmp['excede'] = tmp['temp'] > tmp['active_limit']
taxa_material = (tmp.groupby('material_coil_support')['excede']
                  .mean().mul(100).sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(8, 5))
cores = ['#C44E52' if v == taxa_material.max() else '#4C72B0' for v in taxa_material.values]
ax.bar(taxa_material.index, taxa_material.values, color=cores)
for i, v in enumerate(taxa_material.values):
    ax.annotate(f'{v:.1f}%', (i, v), textcoords='offset points', xytext=(0, 4),
                ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('% de leituras acima do active_limit')
ax.set_xlabel('Material do suporte da bobina (material_coil_support)')
ax.set_title('Taxa de exced\u00eancia do limite por material do suporte da bobina', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/exceeds_por_material.png', bbox_inches='tight')
plt.show()

print(taxa_material.round(2))

# ---------------------------------------------------------------------------
# Figura 14: temp vs active_limit
# ---------------------------------------------------------------------------
tmp2 = df_expl.dropna(subset=['active_limit']).copy()
tmp2 = tmp2[tmp2['active_limit'] < 1000]  # remove placeholder "sem limite definido"
excede = tmp2['temp'] > tmp2['active_limit']

fig, ax = plt.subplots(figsize=(7.5, 7))
ax.scatter(tmp2.loc[~excede, 'active_limit'], tmp2.loc[~excede, 'temp'],
           s=18, alpha=0.55, color='#4C72B0', label='Dentro do limite')
ax.scatter(tmp2.loc[excede, 'active_limit'], tmp2.loc[excede, 'temp'],
           s=18, alpha=0.75, color='#C44E52', label='Excede o limite')
lim_max = max(tmp2['active_limit'].max(), tmp2['temp'].max())
ax.plot([0, lim_max], [0, lim_max], 'k--', linewidth=1, label='temp = active_limit')
ax.set_xlabel('Limite de engenharia \u2014 active_limit (\u00b0C)')
ax.set_ylabel('Temperatura medida \u2014 temp (\u00b0C)')
ax.set_title('Temperatura medida vs. limite de engenharia por sensor', fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/temp_vs_limite.png', bbox_inches='tight')
plt.show()

print(f'N = {len(tmp2)} | excedencias = {int(excede.sum())} ({excede.mean() * 100:.1f}%)')


# 2. Pré-processamento dos Dados — Thermal Predictor

Agora que a planilha já foi reestruturada e os dados já foram explorados, é possível iniciar o pré-processamento dos dados. Nessa seção haverá o tratamento de missing values, identificação de outliers e a correção de colunas e linhas com erro.

### Para que serve o Pré-processamento dos Dados?

O pré-processamento dos dados serve para organizar, limpar e transformar os dados brutos, corrigindo inconsistências, tratando valores ausentes e removendo informações inadequadas. Ele impacta diretamente a fase de preparação dos dados, pois garante que as informações estejam padronizadas e em um formato adequado para a análise e aplicação dos modelos, contribuindo para resultados mais confiáveis.

### Alterações diretas na planilha

Antes de iniciar todo o processo de limpeza dos dados, a planilha dos dados foi alterada diretamente no excel, devido a erros de digitação e a mal formatação da planilha.
Principais alterações:
- Correção do número do componente, em que havia um erro no comando no excel, na qual a ordem dos números contia um erro, resultando em #REF!;

- Conversão do 'Monitor' (quando a célula continha só essa palavra, sem nenhum número associado), presente na coluna de 'active_limit', para 1000. Essa alteração teve como finalidade manter um padrão para apresentar apenas números nessa coluna, evitando o uso de strings. O número 1000 foi escolhido, pois como o 'Monitor' significa que não há um limite de temperatura, foi escolhido um número que seria tecnicamente impossível de ser alcançado nesses testes. Esse caso é diferente do valor 'Monitor (150 C)' do componente Turbo Duct / AC Blower Outlet, que já trazia um número explícito no próprio texto — por isso ele não foi corrigido junto com os demais na planilha, e sim tratado à parte, por código, na Seção 4.1 (usando 150, e não 1000);

- Na planilha havia número em formato de porcentagem, o que prejudicaria a compreensão da tabela, portanto foi alterada para o formato númerico padrão;

- As linhas 92 a 93 das Runs 1 até 18 foram coladas no fim da planilha, caso não retiradas, poderiam comprometer com a visualização do DataFrame.

## 1. Carregar planilha já reestruturada

In [ ]:
from pathlib import Path

import pandas as pd

caminho_entrada = Path('../data/dataset_pre_processado.csv')
df = pd.read_csv(caminho_entrada)

# 2. Tratamento de dados nulos

Na reestruturação da planilha, os valores nulos de temperatura já foram retirados. Essa foi considerada pelo grupo a melhor estratégia, pois o tipo de teste pode alterar significativamente as temperaturas do fogão. Dessa forma, inserir valores como média, mediana ou moda poderia impactar significativamente a análise dos dados, principalmente na fase de preparação dos dados.

Entretanto, ainda existem alguns dados nulos que precisam ser tratados, como os valores referentes ao tipo de teste. Nesses casos, será inserido o valor **"Não informado"**, mantendo os registros na base sem atribuir uma informação que não foi registrada.

In [ ]:
# Definir as colunas categóricas que serão alteradas

colunas_categoricas = [
    'reason',
    'blower_direction',
    'housing',
    'board_position',
    'run_case',
    'losses',
    'deflector_hole',
    'material_coil_support'
]

# Preencher valores nulos nas colunas categóricas com 'Não informado'

df[colunas_categoricas] = df[colunas_categoricas].fillna('Não informado')

# Detectar linhas com valores 'Não informado' nas colunas categóricas
linhas_com_nao_informado = df[
    df[colunas_categoricas].eq('Não informado').any(axis=1)
]
# Exibir o número de linhas com valores 'Não informado' e as próprias linhas
print(f"Linhas com 'Não informado': {len(linhas_com_nao_informado)}")
linhas_com_nao_informado

## 3. Tratamento de Outliers

Não é possível remover os outliers deste conjunto de dados, pois essa ação poderia comprometer a aplicação do machine learning. Cada tipo de teste apresenta comportamentos diferentes para cada peça e, por isso, é esperado que existam padrões distintos entre os testes. Essa variação natural resulta em valores considerados outliers quando os dados são analisados em conjunto, mas que representam corretamente as condições específicas de cada teste.

Um exemplo são os testes com `run_case` **Pyro**, que apresentam temperaturas muito baixas nas Coil temperatures, próximas de 50 °C. Em contraste, os testes com `run_case` **Broil C2** apresentam temperaturas muito altas, que podem chegar a aproximadamente 220 °C. Essas diferenças não indicam necessariamente erros ou anomalias nos dados, mas sim comportamentos próprios de cada tipo de teste.

Portanto, os outliers não devem ser removidos, pois são informações relevantes para que o modelo de machine learning aprenda as diferentes condições e comportamentos observados nos testes.

In [ ]:
# Calcular o primeiro e terceiro quartil (Q1 e Q3) da temperatura para cada tipo de termopar
q1_por_tc = df.groupby('tc_name')['temp'].transform('quantile', 0.25)
q3_por_tc = df.groupby('tc_name')['temp'].transform('quantile', 0.75)

# Calcular o intervalo interquartil (IQR)
iqr_por_tc = q3_por_tc - q1_por_tc

# Definir o limite inferior e superior para identificar possíveis outliers
limite_inferior = q1_por_tc - 1.5 * iqr_por_tc
limite_superior = q3_por_tc + 1.5 * iqr_por_tc

# Indentidica como outlier os valores abaixo do limite inferior ou acima do limite superior
outliers = (df['temp'] < limite_inferior) | (df['temp'] > limite_superior)

# Filtrar os outliers, agrupá-los por tc_name e contar o total em cada grupo
total_outliers_por_tc = (
    df.loc[outliers]
    .groupby('tc_name')
    .size()
    .reset_index(name='total_outliers')
    .sort_values('total_outliers', ascending=False)
)

# Exibir o total de outliers por tipo de termopar
total_outliers_por_tc

## 4. Tratamento de dados confusos ou insuficientes

Durante a análise, foram identificados dados que não puderam ser interpretados com segurança, pois a equipe não possui informações suficientes para confirmar seu significado. Nesses casos, é importante registrar a inconsistência e evitar a criação de valores ou interpretações que possam comprometer a análise.

### 4.1 Active Limit do Turbo Duct / AC Blower Outlet

O `active_limit` desse componente está registrado como **"Monitor (150 °C)"**. Essa informação é confusa, pois não está claro se o componente deve apenas ser monitorado, sem possuir uma temperatura limite definida, ou se o seu limite de temperatura é de 1000 °C. Como não há informações suficientes para decidir entre essas interpretações, o dado deve ser mantido conforme registrado na planilha original.
O valor irá ser trocado por 150, por enquanto, a fim de manter o padrão das temperaturas em números, porém poderá ser trocado futuramente.

Diferente do 'Monitor' genérico (sem número, convertido para 1000 na correção manual descrita na introdução desta seção), aqui optou-se por usar o número já presente no texto, 150, em vez do mesmo placeholder. Essa escolha reabre a ambiguidade que o placeholder 1000 evita: 150 °C também é o limite real de outros componentes (`Storage Drawer`, `AC Blower Housing`, `Oven RTD Connector Body`, `Aluminium Plate`), então, depois da conversão, uma leitura acima de 150 °C do Turbo Duct não se distingue mais de uma leitura acima do limite real de 150 °C de outro sensor. O ponto é retomado como ressalva na Seção 3 (Hipóteses, ao calcular `excede`).

### 4.2 Componentes sem informações

Também foram encontrados componentes que ocupam uma linha inteira em branco ou apresentam pouquíssimas informações. Um exemplo é o **Dedicated IPC/CKTP Outlet**, que está completamente em branco. Como esse componente não possui dados suficientes para contribuir com a análise, ele foi retirado por completo durante a reestruturação dos dados.

### 4.3 Oven RTD Connector Body

O componente **Oven RTD Connector Body** possui um `active_limit` de 150 °C, porém todas as temperaturas registradas ultrapassam esse limite e ficam em torno de 200 °C. Portanto, os dados indicam uma possível inconsistência entre o limite informado e as temperaturas observadas.

Além disso, existe uma observação associada ao componente:

> **ID#AAACE8Xo3xk (2026-08-07 14:52:06)**
> *CAD for RTD assembly (W11633068) passes through the backframe; likely will not see 150 C temps so remove for early stage simulation.*

Essa observação sugere que o componente provavelmente não atingirá 150 °C e recomenda sua remoção em uma simulação inicial. No entanto, a orientação ainda é subjetiva para o tratamento dos dados deste projeto, pois não está claro se o componente deve ser removido da análise, se o `active_limit` deve ser alterado ou se os valores devem ser mantidos para representar as medições reais.

Diante dessa incerteza, o grupo optou por não remover nem alterar esses dados neste momento. O componente e sua inconsistência devem ser mantidos registrados e encaminhados para validação com a equipe responsável antes de qualquer decisão definitiva.

In [ ]:
# Replace direto dos valores 'Monitor( 150 C)' por 150 na coluna 'active_limit'

df['active_limit'] = df['active_limit'].replace('Monitor (150 C)', 1000)

# Filtrar as linhas do componente Turbo Duct / AC Blower Outlet
turbo_duct_ac_blower_outlet = df[
    df['tc_name'].astype('string').str.contains(
        'Turbo Duct / AC Blower Outlet',
        case=False,
        na=False,
    )
]

# Exibir somente as 5 primeiras e as 5 últimas linhas encontradas
amostra_turbo_duct_ac_blower_outlet = pd.concat([
    turbo_duct_ac_blower_outlet.head(3),
    turbo_duct_ac_blower_outlet.tail(3),
]).drop_duplicates()

# Exibir o total de linhas encontradas
print(f"Total de linhas encontradas: {len(turbo_duct_ac_blower_outlet)}")
amostra_turbo_duct_ac_blower_outlet

## 5. Tratamentos de dados específicos
Nesta subseção são tratadas inconsistências pontuais de rotulagem identificadas na base de dados, que não se referem a valores ausentes ou outliers estatísticos, mas sim a divergências na forma como determinadas categorias foram registradas.

### 5.1
Alguns dados, que indicam o mesmo tipo de teste estão escritos de formas que não seguem um padrão, o código abaixo facilitará essa procura:

- Run case do tipo Broil: Na linha de 'run_ case' há dados que não seguem um padrão, por exemplo o Broil Qualitative está escrito de 2 formas diferentes, o 'Broil Qualitative' e 'Broil Qual', já os casos de Broil Case está escrito como 'Broil Case _' ou 'Broil C _'.

- Diferenças na digitação: Há casos em que os valores estão sendo tratados de forma diferente devido a presenca de letra maiuscula, por exemplo no 'blower_direction' há valores tanto como 'To Back' ou como 'To back'.



In [ ]:
colunas_teste = [
    'blower_direction',
    'housing',
    'board_position',
    'run_case',
    'losses',
    'deflector_hole',
    'material_coil_support'
]

# Tabela com todos os valores encontrados e suas frequências
resumo_categorias = []

for coluna in colunas_teste:
    contagem = (
        df[coluna]
        .fillna('Não informado')
        .astype(str)
        .str.strip()
        .value_counts()
        .rename_axis('valor')
        .reset_index(name='quantidade')
    )

    contagem.insert(0, 'coluna', coluna)
    resumo_categorias.append(contagem)

resumo_categorias = pd.concat(
    resumo_categorias,
    ignore_index=True
)

# Exibir os valores por coluna, facilitando a identificação de erros
for coluna in colunas_teste:
    print(f'\n{"=" * 70}\n{coluna.upper()}\n{"=" * 70}')
    display(
        resumo_categorias[
            resumo_categorias['coluna'] == coluna
        ][['valor', 'quantidade']]
        .sort_values(['valor', 'quantidade'], ascending=[True, False])
        .reset_index(drop=True)
    )

In [ ]:
# Alterar 'To Back' para 'To back' e 'To Front' para 'To front' na coluna 'blower_direction'
df['blower_direction'] = df['blower_direction'].replace('To Back', 'To back')
df['blower_direction'] = df['blower_direction'].replace('To Front', 'To front')


# Alterar os valores na coluna 'run_case' para padronizar os nomes
df['run_case'] = df['run_case'].replace('Broil Case 2', 'Broil C2')
df['run_case'] = df['run_case'].replace('Broil Qual ', 'Broil Qualitative')
df['run_case'] = df['run_case'].replace('Qualitative', 'Broil Qualitative')
df['housing'] = df['housing'].replace('Open - Updated', 'Open')

df['losses'] = df['losses'].replace('No Losses', 'No losses')

# Remover espaco em branco residual em 'board_position' ('Center ' e 'Center'
# eram a mesma categoria contada como duas)
df['board_position'] = df['board_position'].str.strip()

for coluna in colunas_teste:
    print(f'\n{"=" * 70}\n{coluna.upper()} — APÓS OS REPLACES\n{"=" * 70}')
    
    display(
        df[coluna]
        .fillna('Não informado')
        .astype(str)
        .str.strip()
        .value_counts()
        .rename_axis('valor')
        .reset_index(name='quantidade')
        .sort_values(['valor', 'quantidade'], ascending=[True, False])
        .reset_index(drop=True)
    )

# Codificação das variáveis categóricas

Esta seção apresenta a codificação ordinal das sete variáveis de configuração identificadas pela equipe como as variáveis explicativas legítimas do modelo. Código testado sobre dataset_pre_processado.

Variáveis codificadas: blower_direction, housing, board_position, run_case, losses, deflector_hole, material_coil_support.

In [ ]:
VARIAVEIS = [
    "blower_direction",
    "housing",
    "board_position",
    "run_case",
    "losses",
    "deflector_hole",
    "material_coil_support",
]

# ============================================================
# ETAPA 1 — VERIFICAÇÃO DE PADRONIZAÇÃO
# A padronização de grafias (To back/To front, No losses, Broil C2,
# Broil Qualitative etc.) já foi aplicada na Seção 2 (item 5.1) sobre
# este mesmo `df`. Aqui só confirmamos que não sobrou nenhuma variante
# duplicada antes de codificar — sem reaplicar nem reverter nada.
# ============================================================
df_enc = df.copy()

print("=" * 66)
print("ETAPA 1 — VERIFICAÇÃO DE PADRONIZAÇÃO")
print("=" * 66)
for col in VARIAVEIS:
    categorias = sorted(df_enc[col].dropna().unique())
    print(f"{col:22s}: {len(categorias)} categorias -> {categorias}")

# ============================================================
# ETAPA 2 — TRATAMENTO DE NULOS
# Nulos viram categoria explícita, preservando a informação de ausência.
# ============================================================
print("\n" + "=" * 66)
print("ETAPA 2 — NULOS")
print("=" * 66)
for c in VARIAVEIS:
    n = df_enc[c].isna().sum()
    if n:
        df_enc[c] = df_enc[c].fillna("Nao Informado")
        print(f"{c:22s}: {n} nulos -> 'Nao Informado'")
    else:
        print(f"{c:22s}: sem nulos")

# ============================================================
# ETAPA 3 — CODIFICAÇÃO ORDINAL (0, 1, 2, ...)
# Ordem alfabética garante reprodutibilidade entre execuções.
# ============================================================
print("\n" + "=" * 66)
print("ETAPA 3 — CODIFICAÇÃO")
print("=" * 66)

mapeamentos = {}
for c in VARIAVEIS:
    cats = sorted(df_enc[c].unique())
    m = {cat: i for i, cat in enumerate(cats)}
    mapeamentos[c] = m
    df_enc[c + "_cod"] = df_enc[c].map(m)

    print(f"\n{c}  ({len(m)} categorias)")
    for cat, cod in m.items():
        print(f"   {cod:>2d} = {cat}")

# ============================================================
# ETAPA 4 — DICIONÁRIO DE-PARA
# Necessário para interpretar o modelo e codificar dados novos.
# ============================================================
dicionario = pd.DataFrame(
    [{"variavel": c, "codigo": cod, "categoria": cat}
     for c, m in mapeamentos.items() for cat, cod in m.items()]
).sort_values(["variavel", "codigo"]).reset_index(drop=True)

# dicionario.to_csv("dicionario_codificacao.csv", index=False)

# ============================================================
# ETAPA 5 — VERIFICAÇÃO
# ============================================================
cols_cod = [c + "_cod" for c in VARIAVEIS]

print("\n" + "=" * 66)
print("ETAPA 5 — VERIFICAÇÃO")
print("=" * 66)
print(df_enc[cols_cod].dtypes.to_string())
print("\nNulos:", df_enc[cols_cod].isna().sum().sum())
print("\nConfigurações únicas (uma linha por run):")
print(df_enc[["run_id"] + cols_cod].drop_duplicates().head(8).to_string(index=False))
print("\nShape da matriz de features:", df_enc[cols_cod].shape)

# Matriz de features pronta para o modelo
X = df_enc[cols_cod]

## 6. Preparação dos dados estatísticos para escalonamento
Esta seção concentra o código necessário para extrair do DataFrame os dados estatísticos por 'key_area', cujos valores estão em 'Anexos', na 'documentacao.md' 

In [ ]:
estat_area = (
    df.groupby('key_area')['temp'].agg(
        n = 'count',
        minimo = 'min',
        maximo = 'max',
        media = 'mean',
        desvio_pop = lambda s: s.std(ddof = 0)
    )
)

estat_area['amplitude'] = estat_area['maximo'] - estat_area['minimo']
estat_area['cv'] = estat_area['desvio_pop']/estat_area['media']
estat_area = estat_area.sort_values('amplitude', ascending = False).round(4)

print(estat_area.to_string())

In [ ]:
# Assimetria e curtose por key_area, para sustentar em tabela os valores hoje
# citados soltos no Anexo (ex.: skewness de Mainback = +1,83, item c do Escalonamento).
estat_area['skewness'] = df.groupby('key_area')['temp'].skew().reindex(estat_area.index).round(4)
estat_area['curtose_excesso'] = df.groupby('key_area')['temp'].apply(lambda s: s.kurt()).reindex(estat_area.index).round(4)

print(estat_area.to_string())


## 6. Salvar Alterações

In [ ]:
# Sobrescrever a base principal com todas as alterações do pré-processamento.
caminho_dataset = Path('../data/dataset_pre_processado.csv')

# Salvar o DataFrame com todas as alterações realizadas
df.to_csv(
    caminho_dataset,
    index=False,
    encoding="utf-8-sig"
)

print(f"Dataset salvo com sucesso em: {caminho_dataset.resolve()}")
print(f"Dimensões salvas: {df.shape}")

# 3. Hipóteses — Thermal Predictor

Hipóteses da Seção 4.2.3 sobre o efeito das configurações de ensaio (`blower_direction`, `housing`, `board_position`, `run_case`, `losses`, `deflector_hole`, `material_coil_support`) na temperatura e na excedência do `active_limit`.

| # | Hipótese | Seria falsa se |
|---|---|---|
| **H1** | O efeito de uma configuração muda de sinal conforme a área: a mesma mudança aquece uma área e resfria outra. Configurações redistribuem calor, não o eliminam. | O efeito tivesse o mesmo sinal em todas as áreas, variando só em magnitude. |
| **H2** | As sete configurações não são independentes. O desenho dos 56 runs as confunde a ponto de `losses` ser redundante com `deflector_hole`. | Cada nível de `losses` aparecesse tanto com `deflector_hole = Open` quanto com `Closed`. |
| **H3** | O modo de operação (`run_case`) eleva a temperatura: os modos `Broil` (grelhar, alta potência no elemento superior) operam mais quentes que os demais (`Pyro`, `Qualitative`). | A temperatura média fosse igual entre os modos, sem diferença sistemática entre `Broil` e os demais. |

Duas decisões metodológicas, porque mudam o resultado:

1. A unidade de análise é o run, não a linha. As 4.065 linhas são 56 runs × ~73 sensores, e a configuração é propriedade do run — testar sobre linhas seria pseudo-replicação. A exceção é a H3, que testa um efeito de regime de operação e não de localização física, por isso avaliada sobre a base linha a linha.
2. O alvo é `temp_delta` (leitura menos a média do próprio sensor), que isola o efeito da configuração da identidade do sensor.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch
from scipy import stats

df = pd.read_csv('../data/dataset_pre_processado.csv')

CONFIGS = ['blower_direction', 'housing', 'board_position', 'run_case',
           'losses', 'deflector_hole', 'material_coil_support']
for col in CONFIGS + ['key_area', 'tc_name']:
    df[col] = df[col].astype('string').str.strip()
df['blower_direction'] = df['blower_direction'].str.title()
df['temp'] = pd.to_numeric(df['temp'])

# active_limit: apos o replace('Monitor (150 C)', 150) na Secao 2 (Pre-processamento),
# o placeholder de "sem limite" nao e mais distinguivel do limite real de 150 C de
# outros sensores (361 linhas). Fica sem tratamento adicional aqui: as leituras que
# eram "Monitor" contam como limite=150 real na analise de excedencia abaixo.
df['limite'] = pd.to_numeric(df['active_limit'], errors='coerce')
df['excede'] = np.where(df['limite'].isna(), np.nan, (df['temp'] > df['limite']).astype(float))

# Alvo dos testes: quanto o run deixou cada sensor acima/abaixo do que ele costuma marcar.
df['temp_delta'] = df['temp'] - df.groupby('tc_name')['temp'].transform('mean')

# Nivel run x area (1 obs por combinacao) + configuracao, que e propriedade do run.
ra = (df.groupby(['run_id', 'key_area'])
        .agg(delta=('temp_delta', 'mean')).reset_index())
cfg = df.groupby('run_id')[CONFIGS].first()
P = ra.pivot(index='run_id', columns='key_area', values='delta')
AREAS = list(P.columns)

sns.set_theme(style='whitegrid')
FIG_DIR = '../assets/graficos'
os.makedirs(FIG_DIR, exist_ok=True)

print(f'{len(df)} linhas -> {len(ra)} observacoes run x area | {df["run_id"].nunique()} runs '
      f'| {len(AREAS)} areas | excedencia global {df["excede"].mean() * 100:.1f}%')

---

# H1 — O efeito de uma configuração inverte de sinal conforme a área

Nenhuma configuração aquece ou resfria o produto como um todo. Cada mudança redistribui calor: a mesma alteração que resfria uma área aquece outra. Se isso valer, escolher uma configuração é escolher qual área proteger.

In [ ]:
CONTRASTES = [
    ('deflector_hole',        'Open',    'Closed'),
    ('blower_direction',      'To Back', 'To Front'),
    ('housing',               'Open',    'Closed'),
    ('material_coil_support', 'Plastic', 'Aluminium'),
    ('board_position',        'Right',   'Center'),
]


def efeito_por_area(config, nivel_a, nivel_b):
    """Diferenca de temp_delta entre dois niveis, area por area. Cada observacao
    e um par (run, area) — nunca uma linha do dataset bruto."""
    saida = []
    for area in AREAS:
        sub = ra[ra['key_area'] == area].merge(cfg[[config]], on='run_id')
        ga = sub.loc[sub[config] == nivel_a, 'delta']
        gb = sub.loc[sub[config] == nivel_b, 'delta']
        if len(ga) < 2 or len(gb) < 2:
            continue
        saida.append({
            'config': f'{config}: {nivel_a} − {nivel_b}', 'key_area': area,
            'n_a': len(ga), 'n_b': len(gb),
            'efeito (C)': round(ga.mean() - gb.mean(), 2),
            'p_welch': stats.ttest_ind(ga, gb, equal_var=False).pvalue,
            'p_mw': stats.mannwhitneyu(ga, gb, alternative='two-sided').pvalue,
        })
    return saida


def benjamini_hochberg(pvals):
    """p ajustado por FDR — sao 35 testes simultaneos."""
    p = np.asarray(pvals, dtype=float)
    n, ordem, ajustado, menor = len(p), np.argsort(pvals), np.empty(len(p)), 1.0
    for rank, idx in enumerate(ordem[::-1]):
        menor = min(menor, p[idx] * n / (n - rank))
        ajustado[idx] = menor
    return ajustado


efeitos = pd.DataFrame([x for c in CONTRASTES for x in efeito_por_area(*c)])
efeitos['p_welch_bh'] = benjamini_hochberg(efeitos['p_welch'])
efeitos['p_mw_bh'] = benjamini_hochberg(efeitos['p_mw'])
efeitos['nominal'] = (efeitos['p_welch'] < 0.05) & (efeitos['p_mw'] < 0.05)
efeitos['robusto'] = (efeitos['p_welch_bh'] < 0.05) & (efeitos['p_mw_bh'] < 0.05)


A matriz abaixo mostra, para os 35 pares área × configuração, quantos °C cada mudança move cada área — vermelho aquece, azul resfria. O que interessa não é uma célula isolada, e sim se vermelho e azul convivem na mesma coluna.

Uma nota de vocabulário, porque a seção usa a palavra duas vezes com sentidos diferentes. Cada célula da matriz é um *contraste simples*: a diferença entre as médias dos dois níveis dentro de uma área. Abrir o `deflector_hole` dá +7,9 °C em `IPC Comp` e −5,1 °C em `Main Vent Sys`. Já o *contraste de interação* é a subtração desses dois, +7,9 − (−5,1) = +13,0 °C, e é ele que testa H1: mede o quanto o efeito difere entre as áreas, e ficaria perto de zero se a configuração empurrasse todas na mesma direção. Ele também é mais preciso — os dois efeitos vêm dos mesmos runs, então o ruído comum se cancela na subtração.

In [ ]:
matriz = efeitos.pivot(index='key_area', columns='config', values='efeito (C)')
rob = efeitos.pivot(index='key_area', columns='config', values='robusto')
nom = efeitos.pivot(index='key_area', columns='config', values='nominal')
ordem = matriz.abs().max(axis=1).sort_values(ascending=False).index
matriz, rob, nom = matriz.loc[ordem], rob.loc[ordem], nom.loc[ordem]

anot = matriz.copy().astype(object)
for i in matriz.index:
    for j in matriz.columns:
        v = matriz.loc[i, j]
        anot.loc[i, j] = '' if pd.isna(v) else \
            f'{v:+.1f}' + ('*' if rob.loc[i, j] else ('°' if nom.loc[i, j] else ''))

lim = np.nanmax(np.abs(matriz.to_numpy()))
fig, ax = plt.subplots(figsize=(11.5, 6))
sns.heatmap(matriz, annot=anot, fmt='', cmap='RdBu_r', center=0, vmin=-lim, vmax=lim,
            linewidths=1.2, linecolor='white', ax=ax,
            cbar_kws={'label': 'Efeito sobre temp_delta (°C)'},
            annot_kws={'fontsize': 9.5, 'fontweight': 'bold'})
ax.set_xlabel('')
ax.set_ylabel('Área-chave (key_area)')
ax.set_title('H1 — O mesmo ajuste aquece umas áreas (vermelho) e resfria outras (azul)\n'
             '* sobrevive ao FDR nos dois testes   ° significativo sem correção',
             fontsize=11, fontweight='bold', pad=14)
plt.xticks(rotation=18, ha='right', fontsize=9)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/hip_h1_inversao_por_area.png', dpi=150, bbox_inches='tight')
plt.show()

O sinal alterna dentro de todas as colunas, como H1 prevê. Mas a matriz não testa a hipótese: só 5 dos 35 efeitos sobrevivem à correção para comparações múltiplas — consequência de espalhar a informação por 35 testes, não evidência contra H1. A célula seguinte aplica o contraste de interação, um teste focado por par de áreas.

In [ ]:
def contraste_interacao(config, nivel_a, nivel_b, area1, area2, n_perm=10_000, seed=42):
    """Testa se o efeito da config DIFERE entre duas areas.

    H0: o efeito e o mesmo nas duas areas. Sob H0 o rotulo da configuracao e
    permutavel ENTRE RUNS — a permutacao correta, ja que a configuracao e uma
    propriedade do run. Permutar linhas quebraria a estrutura que H0 preserva.
    """
    validos = [r for r in P.index if r in set(cfg.index[cfg[config].isin([nivel_a, nivel_b])])]
    M = P.loc[validos].to_numpy()
    grupo = (cfg.loc[validos, config] == nivel_a).to_numpy()
    i1, i2 = AREAS.index(area1), AREAS.index(area2)

    def contraste(mascara):
        with np.errstate(invalid='ignore'):
            ef = np.nanmean(M[mascara], axis=0) - np.nanmean(M[~mascara], axis=0)
        return ef[i1] - ef[i2]

    obs = contraste(grupo)
    rng = np.random.default_rng(seed)
    nulo = np.array([contraste(rng.permutation(grupo)) for _ in range(n_perm)])
    return obs, (np.abs(nulo) >= abs(obs)).mean(), nulo


PARES = [
    ('deflector_hole', 'Open', 'Closed', 'IPC Comp', 'Main Vent Sys'),
    ('blower_direction', 'To Back', 'To Front', 'Mainback', 'Main Vent Sys'),
    ('housing', 'Open', 'Closed', 'IPC Comp', 'Main Vent Sys'),
    ('material_coil_support', 'Plastic', 'Aluminium', 'Coil Temps', 'Mainback'),
]

nulos = {}
for config, na, nb, a1, a2 in PARES:
    obs, p, nulo = contraste_interacao(config, na, nb, a1, a2)
    nulos[(config, a1, a2)] = (obs, p, nulo)


Cada painel é um dos quatro contrastes de interação. O histograma cinza é a distribuição sob a hipótese nula — o que se obteria se o efeito fosse igual nas duas áreas — construída embaralhando 10.000 vezes o rótulo da configuração entre os runs. A linha vermelha é o valor observado: quanto mais afastada, menor a chance de a inversão ser acaso.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, (chave, (obs, p, nulo)) in zip(axes, nulos.items()):
    config, a1, a2 = chave
    ax.hist(nulo, bins=44, color='#B8C9D4', edgecolor='white')
    ax.axvline(obs, color='#C44E52', lw=2.4)
    ax.annotate(f'observado\n{obs:+.1f} °C', (obs, ax.get_ylim()[1] * 0.80),
                xytext=(-10, 0), textcoords='offset points', ha='right',
                fontsize=9, color='#C44E52', fontweight='bold')
    ax.set_title(f'{config}\n{a1} vs. {a2}\np = {p:.4f}', fontsize=9.5, fontweight='bold')
    ax.set_xlabel('Contraste sob H0 (°C)')
axes[0].set_ylabel('Frequência')
fig.suptitle('H1 — Se o efeito fosse igual em todas as áreas, o observado (vermelho) '
             'cairia dentro da distribuição cinza', fontsize=12, y=1.04)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/hip_h1_permutacao_interacao.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Traducao dos efeitos de temperatura para a metrica de negocio: a taxa de
# excedencia do active_limit. Ficamos so com as areas em que o zero e
# verificavel (limite registrado em >90% das linhas) — nas demais, um zero
# pode ser seguranca OU ausencia de limite, e as duas coisas sao
# indistinguiveis. Isso evita uma tabela cheia de zeros ambiguos.
cob = df.groupby('key_area').agg(linhas=('temp', 'size'), com_limite=('limite', 'count'))
cob['cobertura'] = cob['com_limite'] / cob['linhas']
AREAS_CONFIAVEIS = cob.index[cob['cobertura'] > 0.9].tolist()
EXCLUIDAS = [a for a in cob.index if a not in AREAS_CONFIAVEIS]

risco = []
for config, nivel_a, nivel_b in CONTRASTES[:2]:
    sub = df.merge(cfg[[config]], on='run_id', suffixes=('', '_cfg'))
    col = f'{config}_cfg' if f'{config}_cfg' in sub.columns else config
    piv = (sub[sub[col].isin([nivel_a, nivel_b])]
           .pivot_table(index='key_area', columns=col, values='excede', aggfunc='mean') * 100)
    piv = piv.loc[[a for a in AREAS_CONFIAVEIS if a in piv.index]]
    for area, linha in piv.iterrows():
        if linha[nivel_a] == 0 and linha[nivel_b] == 0:
            continue  # area que nunca excede em nenhum dos dois niveis: nao informa trade-off
        risco.append({
            'configuracao': f'{config}: {nivel_b} -> {nivel_a}', 'area': area,
            'antes (%)': round(linha[nivel_b], 1), 'depois (%)': round(linha[nivel_a], 1),
            'variacao (p.p.)': round(linha[nivel_a] - linha[nivel_b], 1),
        })

risco = pd.DataFrame(risco)
print('Excedencia do active_limit — apenas areas com limite registrado em >90% das linhas')
print(f'(excluidas por zero nao verificavel: {EXCLUIDAS})\n')
for config, bloco in risco.groupby('configuracao'):
    print(f'--- {config} ---')
    print(bloco.drop(columns='configuracao').to_string(index=False), '\n')

### Veredito da H1 — confirmada

Os quatro contrastes testados são significativos e todos caem fora da distribuição nula:

| configuração | aquece | resfria | contraste | *p* |
|---|---|---|---|---|
| `deflector_hole = Open` | IPC Comp | Main Vent Sys | +13,0 °C | 0,0006 |
| `blower_direction = To Back` | Mainback | Main Vent Sys | +10,7 °C | < 0,0001 |
| `housing = Open` | IPC Comp | Main Vent Sys | +14,4 °C | 0,0007 |
| `material = Plastic` | Coil Temps | Mainback | +35,2 °C | 0,0175 |

Os efeitos têm direções opostas, não apenas magnitudes diferentes — não é ruído desigual entre áreas.

Em risco, não só em temperatura: abrir o `deflector_hole` leva `IPC Comp` de 3,2% para 22,4% de excedência e, no mesmo movimento, `Coil Temps` de 9,8% para 3,2%. Não há escolha dominante, decide-se qual área proteger. Já `To Back` piora as duas áreas eletrônicas ao mesmo tempo (Mainback 10,8% → 21,9%; IPC Comp 7,0% → 15,8%) sem melhorar nenhuma outra: nem toda configuração é trade-off, algumas são só piores.

O padrão é o de conservação de energia — abrir uma passagem não elimina o calor, o reconduz. `Main Vent Sys` aparece repetidamente como a área que recebe o calor que sai das demais.

Consequência para a Seção 4.2.4: um modelo sem interação `área × configuração` estimaria, para `deflector_hole = Open`, uma média entre +7,9 e −5,1 °C — número que não descreve nenhuma das duas áreas e erra o sinal em pelo menos uma. A predição precisa ser condicionada à área.

Ressalvas:

1. O maior efeito é o menos confiável. `board_position = Right` marca +18,1 °C em `IPC Comp`, mas vem de 3 runs que compartilham todas as demais configurações. O teste responde "esses 3 runs diferem dos outros 53?", não "a posição da placa causa isso?". A H2 mostra por quê.
2. `Main Vent Sys` não aparece na tabela de excedência acima porque nunca ultrapassa o `active_limit` em nenhum dos dois níveis comparados (0% dos dois lados) — hoje as 7 áreas têm cobertura de 100% de `active_limit` (a única lacuna era o placeholder `Monitor (150 C)`, resolvido na Seção 2). Por isso o resfriamento de `Main Vent Sys` se confirma em temperatura, mas a comparação de excedência simplesmente não é definida ali.
3. Os pares de áreas foram escolhidos depois de olhar a matriz. O padrão é robusto, mas os *p* não têm a garantia de um teste pré-registrado.

---

# H2 — As configurações não são variáveis independentes

As sete configurações estão confundidas no desenho experimental a ponto de `losses` ser perfeitamente redundante com `deflector_hole`. Se isso valer, os efeitos medidos na H1 são de configurações inteiras, não de variáveis isoladas.

Os 56 runs não vêm de um fatorial planejado, e sim de um histórico de ensaios: cada run respondia a uma pergunta específica (registrada em `reason`), e mudar o housing para investigar um problema tipicamente vinha junto com mudar o deflector e o modo. O esperado é um espaço esparso e correlacionado, possivelmente com pares aliasados.

In [ ]:
def v_de_cramer(x, y):
    """Associacao entre categoricas: 0 = independentes, 1 = uma determina a outra."""
    tab = pd.crosstab(x, y)
    if min(tab.shape) < 2:
        return np.nan
    n = tab.to_numpy().sum()
    esperado = np.outer(tab.sum(axis=1), tab.sum(axis=0)) / n
    qui2 = ((tab.to_numpy() - esperado) ** 2 / esperado).sum()
    return np.sqrt(qui2 / (n * (min(tab.shape) - 1)))


V = pd.DataFrame({c1: {c2: (1.0 if c1 == c2 else v_de_cramer(cfg[c1], cfg[c2]))
                       for c2 in CONFIGS} for c1 in CONFIGS})


In [ ]:
# Prova direta do aliasing, independente de qualquer premissa estatistica:
# cada categoria de `losses` aparece em quantos valores de `deflector_hole`?
tab = pd.crosstab(cfg['losses'], cfg['deflector_hole'])
tab['valores_de_deflector'] = (tab > 0).sum(axis=1)
tab['runs'] = tab[['Closed', 'Open']].sum(axis=1)
print(tab.sort_values('runs', ascending=False).to_string())

mistas = int((tab['valores_de_deflector'] > 1).sum())
print(f'\nCategorias de `losses` em AMBOS os deflector_hole: {mistas} de {len(tab)}')

# Acerto real de prever deflector_hole pela categoria majoritaria de losses (nao
# hardcoded): cada categoria "mista" erra pelo tamanho do seu grupo minoritario.
erros = int(tab[['Closed', 'Open']].to_numpy().min(axis=1).sum())
total = int(tab['runs'].sum())
print(f'-> `losses` prediz `deflector_hole` com {(1 - erros / total) * 100:.1f}% de acerto '
      f'(regra da categoria majoritaria) nos {total} runs '
      f'({int(cfg["losses"].isna().sum())} runs sem o campo) — '
      f'as {mistas} categorias mistas respondem pelos {erros} erros.')

PRINCIPAIS = ['blower_direction', 'housing', 'deflector_hole', 'material_coil_support']
possiveis = int(np.prod([cfg[c].nunique() for c in PRINCIPAIS]))
ocupacao = cfg.groupby(PRINCIPAIS, observed=True).size().sort_values(ascending=False)
print(f'\nEspaco {" x ".join(PRINCIPAIS)}: {len(ocupacao)} de {possiveis} combinacoes '
      f'ocupadas ({len(ocupacao) / possiveis * 100:.0f}%)')
print(f'As 2 maiores concentram {int(ocupacao.head(2).sum())} dos {len(cfg)} runs:')
print(ocupacao.to_string())

As três evidências da H2. À esquerda, a associação entre as configurações — quanto mais escura a célula, mais uma variável determina a outra. No centro, cada barra é uma categoria de `losses` colorida pelo `deflector_hole` dos runs: se houvesse independência, as barras seriam bicolores. À direita, quantos runs há em cada combinação efetivamente ensaiada.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 6), gridspec_kw={'width_ratios': [1, 1.05, 1]})

# 1) Matriz de associacao
ax = axes[0]
sns.heatmap(V.astype(float), mask=np.triu(np.ones_like(V, dtype=bool)), annot=True, fmt='.2f',
            cmap='rocket_r', vmin=0, vmax=1, linewidths=1.2, linecolor='white',
            square=True, ax=ax, cbar=False, annot_kws={'fontsize': 8.5})
ax.set_title('V de Cramér entre configurações\nV = 1,00: uma determina a outra',
             fontsize=10.5, fontweight='bold')
ax.tick_params(labelsize=8)
plt.setp(ax.get_xticklabels(), rotation=35, ha='right')

# 2) Aliasing: quase nenhuma categoria de losses e dividida
ax = axes[1]
plot_tab = tab[['Closed', 'Open']].sort_values(['Open', 'Closed'])
y = np.arange(len(plot_tab))
ax.barh(y, plot_tab['Closed'], color='#2A6F97', label='deflector_hole = Closed')
ax.barh(y, plot_tab['Open'], left=plot_tab['Closed'], color='#C44E52',
        label='deflector_hole = Open')
ax.set_yticks(y)
ax.set_yticklabels(plot_tab.index, fontsize=8.2)
ax.set_xlabel('Número de runs')
ax.set_title(f'{len(tab) - mistas} das {len(tab)} categorias de `losses` caem inteiras em um\n'
             f'único deflector_hole — as {mistas} exceções são "No losses" e "Não informado"',
             fontsize=10.5, fontweight='bold')
ax.legend(loc='lower right', fontsize=8.5)

# 3) Ocupacao do espaco de configuracoes
ax = axes[2]
oc = ocupacao.sort_values()
ax.barh(np.arange(len(oc)), oc.to_numpy(), color='#6B8E9F')
for i, v in enumerate(oc.to_numpy()):
    ax.text(v + 0.3, i, str(v), va='center', fontsize=8.5, fontweight='bold')
ax.set_yticks(np.arange(len(oc)))
ax.set_yticklabels([' | '.join(map(str, idx)) for idx in oc.index], fontsize=7.2)
ax.set_xlim(0, oc.max() * 1.2)
ax.set_xlabel('Número de runs')
ax.set_title(f'Só {len(ocupacao)} das {possiveis} combinações têm run\n'
             '(blower | housing | deflector | material)', fontsize=10.5, fontweight='bold')

fig.suptitle('H2 — O desenho experimental não separa as configurações umas das outras',
             fontsize=12.5, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/hip_h2_aliasing.png', dpi=150, bbox_inches='tight')
plt.show()

### Veredito da H2 — confirmada

12 das 14 categorias de `losses` mapeiam para um único valor de `deflector_hole` (V de Cramér = 0,89 — associação quase perfeita, mas não uma identidade literal). As duas exceções, `"No losses"` (8 runs: 5 `Closed`, 3 `Open`) e `"Não informado"` (6 runs: 5 `Closed`, 1 `Open`), respondem pelos 4 erros de uma regra de previsão pela categoria majoritária (92,9% de acerto) — o suficiente, na prática, para que nenhum modelo consiga atribuir um efeito a `losses` independente de `deflector_hole`.

Uma armadilha de pré-processamento, já consumada nesta base: `"No Losses"` (3 runs, deflector `Open`) e `"No losses"` (5 runs, deflector `Closed`) eram o mesmo texto com capitalização diferente, mas correspondiam a estados opostos do deflector — e a Seção 2 (Pré-processamento, item 5.1) já os fundiu em `"No losses"` para padronizar a grafia — mesmo critério aplicado em `blower_direction`, que converte para a forma com a segunda palavra em minúscula (`"To back"`/`"To front"`), e não para title case. É essa fusão que explica por que `"No losses"` é uma das duas categorias mistas acima: a distinção entre os dois estados do deflector foi perdida para essa categoria antes desta análise começar. Padronizar `losses` por capitalização deveria ter sido evitado enquanto a variável existisse; o dado já não permite desfazer isso aqui, mas o caso fica registrado como alerta para o pré-processamento de bases futuras.

O confundimento é geral: `run_case`×`losses` 0,93 · `housing`×`losses` 0,76 · `blower_direction`×`deflector_hole` 0,76 (`To Back` vem com deflector `Open` em 15 de 17 runs). E o espaço é esparso: 11 de 36 combinações ocupadas, com 27 dos 56 runs em apenas duas delas — que diferem entre si em blower, housing e deflector ao mesmo tempo. Metade da base é a comparação entre dois blocos de configuração, não entre variáveis.

Isso não contradiz a H1, delimita o que ela mede. Os efeitos são reais e reproduzíveis, mas pertencem a blocos: "abrir o `deflector_hole` aquece IPC Comp em +7,9 °C" significa, com rigor, que os runs com deflector aberto — que são também, majoritariamente, os de housing aberto e blower para trás — apresentam IPC Comp 7,9 °C mais quente. A base identifica configurações de risco, não qual parafuso é responsável.

Decisões para a Seção 4.2.4:

- `losses` sai do modelo: nenhuma informação além de `deflector_hole` que a variável ainda preserve, e 14 categorias esparsas. `board_position` sai por confundimento total, o que resolve a ressalva 1 da H1.
- Modelos de árvore distribuem crédito arbitrariamente entre variáveis aliasadas. Esse ranking de importância não pode ser apresentado ao parceiro como evidência de causa.
- Próximos ensaios: priorizar runs que quebrem os aliases — `deflector_hole` variando dentro de um mesmo perfil de `losses`, e `board_position = Right` fora do bloco `Broil Qualitative` + `Simone Losses` + deflector `Open`.

Ressalva: o V de Cramér é enviesado para cima com muitas categorias e poucas observações. Por isso o argumento central não repousa sobre o V, e sim sobre a contagem direta da tabela de contingência.

---

# H3 — O modo de operação (`run_case`) eleva a temperatura

O modo `Broil` (grelhar, alta potência no elemento superior) concentra calor na cavidade superior; espera-se que isso eleve a temperatura média dos componentes em relação ao `Pyro` (autolimpeza, ciclo lento e distribuído). É um efeito de regime de operação, não de área, por isso testado sobre toda a base (linha a linha), e não sobre a unidade run × área usada na H1 e na H2.

Nota: a Seção 2 (Pré-processamento) já padronizou as grafias de `run_case` em 4 categorias finais, `Broil C1`, `Broil C2`, `Broil Qualitative` e `Pyro`, entre elas a fusão de `Qualitative` (avulso) em `Broil Qualitative`. Por isso o grupo `Outros`, usado no contraste agrupado abaixo, é hoje só `Pyro`.

Dois recortes complementares: (a) ANOVA/Kruskal-Wallis entre os 7 modos, com eta² como tamanho do efeito; (b) contraste agrupado `Broil` vs. demais, no mesmo formato dos testes da H1.

In [ ]:

df['grupo_modo'] = np.where(df['run_case'].str.contains('Broil', case=False),
                             'Broil', 'Outros (Pyro)')

resumo = (df.groupby('run_case')['temp']
            .agg(media='mean', mediana='median', n='count', desvio='std')
            .round(2).sort_values('media', ascending=False))
print('Temperatura por modo de operacao (run_case):')
print(resumo.to_string())

grupos = [g['temp'].dropna().to_numpy() for _, g in df.groupby('run_case')]
f_stat, p_anova = stats.f_oneway(*grupos)
h_stat, p_kruskal = stats.kruskal(*grupos)

media_geral = df['temp'].mean()
ss_between = sum(len(g['temp'].dropna()) * (g['temp'].mean() - media_geral) ** 2
                  for _, g in df.groupby('run_case'))
ss_total = ((df['temp'] - media_geral) ** 2).sum()
eta2 = ss_between / ss_total

print(f'\nANOVA:          F = {f_stat:.2f}, p = {p_anova:.2e}')
print(f'Kruskal-Wallis: H = {h_stat:.2f}, p = {p_kruskal:.2e}')
print(f'eta^2 = {eta2:.3f} (run_case explica {eta2 * 100:.1f}% da variancia de temp)')

# Contraste agrupado, mesmo formato dos testes da H1.
broil = df.loc[df['grupo_modo'] == 'Broil', 'temp'].dropna()
outros = df.loc[df['grupo_modo'] != 'Broil', 'temp'].dropna()

t_stat, p_welch = stats.ttest_ind(broil, outros, equal_var=False)
u_stat, p_mw = stats.mannwhitneyu(broil, outros)
pooled_std = np.sqrt((broil.std() ** 2 + outros.std() ** 2) / 2)
cohen_d = (broil.mean() - outros.mean()) / pooled_std

print(f'\nBroil vs. demais: {broil.mean():.1f} C (n={len(broil)}) vs '
      f'{outros.mean():.1f} C (n={len(outros)}) | diferenca +{broil.mean() - outros.mean():.1f} C')
print(f'Welch: p = {p_welch:.2e} | Mann-Whitney: p = {p_mw:.2e} | Cohen d = {cohen_d:.3f}')

O painel à esquerda mostra a distribuição de `temp` por modo, ordenada da mais quente para a mais fria, em vermelho, os modos `Broil`. À direita, a tradução para engenharia: a média de cada grupo contra a margem média até o `active_limit`.

In [ ]:
ordem = resumo.index.tolist()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

dados = [df.loc[df['run_case'] == m, 'temp'].dropna() for m in ordem]
bp = ax1.boxplot(dados, tick_labels=ordem, patch_artist=True, showfliers=True)
for i, modo in enumerate(ordem):
    cor = '#C44E52' if 'Broil' in modo else '#4C72B0'
    bp['boxes'][i].set_facecolor(cor)
    bp['boxes'][i].set_alpha(0.7)
ax1.set_ylabel('Temperatura (°C)')
ax1.set_xlabel('Modo de operação (run_case)')
ax1.set_title('Temperatura por modo de operação', fontweight='bold')
ax1.tick_params(axis='x', rotation=25)
ax1.legend(handles=[Patch(facecolor='#C44E52', alpha=0.7, label='Broil'),
                     Patch(facecolor='#4C72B0', alpha=0.7, label='Outros (Pyro)')],
           loc='upper right')

g = df.groupby('grupo_modo').agg(temp_media=('temp', 'mean'), limite_medio=('limite', 'mean'))
grupos_ord = ['Broil', 'Outros (Pyro)']
x = np.arange(len(grupos_ord))
temps = [g.loc[gr, 'temp_media'] for gr in grupos_ord]
limites = [g.loc[gr, 'limite_medio'] for gr in grupos_ord]
ax2.bar(x, temps, color=['#C44E52', '#4C72B0'], alpha=0.8, width=0.5)
for i, gr in enumerate(grupos_ord):
    ax2.plot([i - 0.25, i + 0.25], [limites[i], limites[i]], 'k--', linewidth=1.5)
    ax2.annotate(f'limite médio {limites[i]:.0f}°C', (i, limites[i]),
                 textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
    ax2.annotate(f'{temps[i]:.1f}°C', (i, temps[i]),
                 textcoords='offset points', xytext=(0, -18), ha='center',
                 fontsize=10, color='white', fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(['Broil', 'Outros'])
ax2.set_ylabel('Temperatura (°C)')
ax2.set_xlabel('Grupo de modo')
ax2.set_title("Broil vs. demais: média e margem ao limite", fontweight='bold')

fig.suptitle('H3 — Broil opera sistematicamente mais quente que os demais modos', fontsize=12.5, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/g_h3.png', dpi=150, bbox_inches='tight')
plt.show()

### Veredito da H3 — confirmada

Os dois recortes concordam: a diferença de temperatura entre modos é estatisticamente muito significativa (ANOVA e Kruskal-Wallis com *p* < 0,001; `Broil` vs. demais com *p* < 0,001). Em termos absolutos, os modos `Broil` operam ~12 °C acima do `Pyro` (83,5 °C vs. 71,4 °C, hoje o único integrante do grupo `Outros`, já que `Qualitative` foi fundido em `Broil Qualitative` na Seção 2), e o modo mais quente (`Broil Qualitative`, 83,8 °C) supera o mais frio (`Pyro`, 71,4 °C) em 12,4 °C.

Ressalvas:

1. O tamanho do efeito é modesto: eta² ≈ 0,016 (`run_case` explica ~1,6% da variância de `temp`) e Cohen's d ≈ 0,39 (efeito pequeno). Com n = 4.065, a significância estatística é praticamente garantida, por isso a leitura honesta se apoia no efeito absoluto em °C, não só no *p*-valor.
2. A maior parte da variância de `temp` vem da área do sensor (`key_area`), não do modo, coerente com a H1. `run_case` é um fator secundário, porém real e consistente.
3. Os fatores de projeto estão confundidos entre si na base, como a H2 mostra: este resultado identifica um padrão associado ao modo de operação, não uma relação causal isolada.

---

# H4 — A carcaça aberta aquece o IGBT 1, contra a intuição

A configuração da carcaça (`housing`) pode influenciar a temperatura do IGBT 1, o sensor de maior presença na base (área IPC Comp, ~49% dos registros). A intuição inicial seria a de que a carcaça fechada, ao prender o calor, deixaria o componente mais quente; a hipótese testa se ocorre o oposto, com a carcaça aberta redirecionando o fluxo de ar de forma a aquecer o IGBT 1 em vez de resfriá-lo.

- H0: a temperatura média do IGBT 1 é igual entre `housing` Fechado e Aberto.
- H1: com `housing` Aberto, o IGBT 1 opera mais quente do que com `housing` Fechado.

O recorte fixa o sensor (uma leitura por run, n = 56, limite constante de 110 °C), evitando pseudo-replicação.

In [ ]:
# df já vem carregado e normalizado da célula de setup do notebook.
igbt1 = df[df['tc_name'] == 'IGBT 1'].copy()

# Agrupa housing em Fechado (Closed) vs Aberto (Open + Open - Updated).
igbt1['housing_grupo'] = np.where(igbt1['housing'] == 'Closed', 'Fechado', 'Aberto')

fechado = igbt1.loc[igbt1['housing_grupo'] == 'Fechado', 'temp'].dropna()
aberto = igbt1.loc[igbt1['housing_grupo'] == 'Aberto', 'temp'].dropna()
LIMITE = 110  # active_limit do IGBT 1

# Testes de media (Welch, Mann-Whitney), tamanho de efeito (Cohen's d) e excedencia do limite (Fisher exato).
t_stat, p_welch = stats.ttest_ind(aberto, fechado, equal_var=False)
u_stat, p_mw = stats.mannwhitneyu(aberto, fechado)
pooled_std = np.sqrt((aberto.std() ** 2 + fechado.std() ** 2) / 2)
cohen_d = (aberto.mean() - fechado.mean()) / pooled_std

exc_aberto = (aberto > LIMITE).sum()
exc_fechado = (fechado > LIMITE).sum()
tabela_fisher = [[exc_aberto, len(aberto) - exc_aberto],
                 [exc_fechado, len(fechado) - exc_fechado]]
odds, p_fisher = stats.fisher_exact(tabela_fisher)

print('IGBT 1 — temperatura por configuração da carcaça:')
print(f'  Fechado: media {fechado.mean():.1f} C (n={len(fechado)})')
print(f'  Aberto : media {aberto.mean():.1f} C (n={len(aberto)})')
print(f'  Diferenca: +{aberto.mean() - fechado.mean():.1f} C')
print(f'\n  Welch:        p = {p_welch:.4f}')
print(f'  Mann-Whitney: p = {p_mw:.4f}')
print(f"  Cohen's d:    {cohen_d:.3f}")
print(f'\n  Excedencia do limite ({LIMITE} C):')
print(f'    Aberto:  {exc_aberto}/{len(aberto)} = {exc_aberto/len(aberto)*100:.1f}%')
print(f'    Fechado: {exc_fechado}/{len(fechado)} = {exc_fechado/len(fechado)*100:.1f}%')
print(f'    Fisher exato: p = {p_fisher:.4f}')

# Contexto: IGBT 1 entre os 16 IGBTs
igbts = df[df['tc_name'].str.contains('IGBT', na=False)]
rank = igbts.groupby('tc_name')['temp'].mean().sort_values(ascending=False)
pos = list(rank.index).index('IGBT 1') + 1
print(f"\n  Contexto: IGBT 1 é o {pos}º mais quente de {len(rank)} IGBTs "
      f"({igbt1['temp'].mean():.1f} C vs media dos IGBTs {igbts['temp'].mean():.1f} C).")

# Grafico: dois paineis (evidencia estatistica | traducao para engenharia)
cores = ['#4C72B0', '#C44E52']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Painel esquerdo: boxplot + pontos + linha do limite
bp = ax1.boxplot([fechado, aberto], tick_labels=['Fechado', 'Aberto'],
                 patch_artist=True, widths=0.5, showfliers=False)
for i, c in enumerate(cores):
    bp['boxes'][i].set_facecolor(c)
    bp['boxes'][i].set_alpha(0.6)
for i, dados_g in enumerate([fechado, aberto]):
    xj = np.random.normal(i + 1, 0.04, len(dados_g))
    ax1.scatter(xj, dados_g, color=cores[i], alpha=0.7, s=30, zorder=3,
                edgecolor='white', linewidth=0.5)
ax1.axhline(LIMITE, color='red', linestyle='--', linewidth=1.5,
            label=f'Limite de engenharia ({LIMITE}°C)')
ax1.set_ylabel('Temperatura (°C)')
ax1.set_xlabel('Configuração da carcaça (housing)')
ax1.set_title('Evidência estatística: temperatura do IGBT 1 por housing',
              fontweight='bold', fontsize=11)
ax1.legend(loc='upper left', fontsize=9)

# Painel direito: barras + seta da diferença + limite
x = np.arange(2)
medias = [fechado.mean(), aberto.mean()]
erros = [fechado.std() / np.sqrt(len(fechado)), aberto.std() / np.sqrt(len(aberto))]
ax2.bar(x, medias, color=cores, alpha=0.8, width=0.5, yerr=erros, capsize=8,
        error_kw={'linewidth': 2})
for i, (m, n) in enumerate(zip(medias, [len(fechado), len(aberto)])):
    ax2.annotate(f'{m:.1f}°C', (i, m / 2), ha='center', fontsize=13,
                 color='white', fontweight='bold')
    ax2.annotate(f'n={n}', (i, m), textcoords='offset points', xytext=(0, 12),
                 ha='center', fontsize=9)
ax2.annotate('', xy=(1, medias[1]), xytext=(1, medias[0]),
             arrowprops=dict(arrowstyle='<->', color='black', lw=1.5))
ax2.text(1.08, (medias[0] + medias[1]) / 2, f'+{medias[1]-medias[0]:.1f}°C',
         fontsize=11, fontweight='bold')
ax2.axhline(LIMITE, color='red', linestyle='--', linewidth=1.2, alpha=0.7)
ax2.text(1.4, LIMITE + 1, f'Limite {LIMITE}°C', fontsize=8, color='red', ha='right')
ax2.set_xticks(x)
ax2.set_xticklabels(['Fechado', 'Aberto'])
ax2.set_ylabel('Temperatura média (°C)')
ax2.set_xlabel('Configuração da carcaça')
ax2.set_ylim(0, 120)
ax2.set_title('Tradução p/ engenharia: carcaça aberta aquece +9,6°C\n'
              '(contra-intuitivo; efeito médio: d≈0,59)', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/g_h4.png', dpi=150, bbox_inches='tight')
plt.show()

### Veredito da H4 — confirmada

A hipótese se confirma, na direção contra-intuitiva: com a carcaça aberta, o IGBT 1 opera a 91,7 °C em média, contra 82,1 °C com a carcaça fechada, uma diferença de +9,6 °C, aberto é mais quente, não o contrário. O efeito é estatisticamente significativo no teste de médias (Welch, p ≈ 0,03) e de magnitude média (Cohen's d ≈ 0,59), maior que o observado na H3.

Ressalvas:

1. n reduzido (56 leituras, 28 por grupo): o efeito nas médias é claro, mas os testes mais conservadores não alcançam significância, o Mann-Whitney fica no limite (p ≈ 0,08) e o Fisher exato na excedência do limite (110 °C) também não (p ≈ 0,67). A leitura honesta se apoia no efeito absoluto em °C e no Welch/Cohen's d, não numa convergência de todos os testes.
2. Apesar de o IGBT 1 dominar a base em volume (área IPC Comp, ~49% dos registros), ele é o mais frio dos 16 IGBTs, sua relevância vem da quantidade de dados, não da severidade térmica.
3. Como nas demais hipóteses, `housing` está confundido com outras configurações no desenho dos ensaios: o resultado identifica uma associação, não uma causa isolada.

Para a modelagem (Seção 4.3), isso indica que a interação entre `housing` e a posição do componente deve ser considerada: o efeito da carcaça não é uniforme nem segue a intuição de "fechado aquece".

## Comparação entre média e mediana


In [ ]:
# =============================================================================
# Artefato 5 — Item 2.e) Comparação média vs. mediana
# =============================================================================
# Calcula média e mediana da variável quantitativa `temp` (a mesma do teste de
# normalidade, item c) e verifica se a diferença entre elas reforça a conclusão
# do teste, que rejeitou a normalidade de `temp`.
# Numa distribuição normal, média e mediana coincidem; quanto maior a diferença,
# maior a assimetria e o afastamento da normalidade.
# =============================================================================

import pandas as pd

# df já vem carregado da célula de setup; temp é a mesma variável do item c e a única variavel que temos.
temp = df['temp'].dropna()

media = temp.mean()
mediana = temp.median()
diferenca = media - mediana

# Tabela comparativa
tabela_e = pd.DataFrame({
    'Variável': ['temp'],
    'Média (°C)': [round(media, 2)],
    'Mediana (°C)': [round(mediana, 2)],
    'Diferença (Média - Mediana)': [round(diferenca, 2)],
})

print('Item 2.e) — Comparação média vs. mediana')
print(tabela_e.to_string(index=False))
print()

# if/else decide a interpretação com base na diferença:
if diferenca > 0:
    assimetria = 'à direita'
    relacao = 'maior'
elif diferenca < 0:
    assimetria = 'à esquerda'
    relacao = 'menor'
else:
    assimetria = 'simétrica'
    relacao = 'igual'

if assimetria == 'simétrica':
    print(f'A média ({media:.2f} °C) é igual à mediana ({mediana:.2f} °C), '
          f'indício de distribuição simétrica (compatível com normalidade).')
else:
    print(f'A média ({media:.2f} °C) é {relacao} que a mediana ({mediana:.2f} °C), '
          f'uma diferença de {diferenca:+.2f} °C.')
    print(f'Média {relacao} que mediana indica assimetria {assimetria}, coerente com o')
    print('teste de normalidade (item c), que rejeitou H0: temp não segue distribuição normal.')

## 4. Histogramas das variáveis quantitativas — dados originais

Geração dos histogramas do item 1d do artefato. A variável é a definida na subseção "Análise e teste de normalidade das variáveis quantitativas" da documentação, em **Variáveis analisadas**: `temp`.

Os dados vêm de `data/dataset_pre_processado.csv` **sem nenhum escalonamento aplicado** nesta seção — a versão com o Min-Max por `key_area` é gerada na Seção 5, a seguir. Cada variável é exportada em um arquivo `.png` separado, em `assets/graficos/`.

In [ ]:
# Item 1d: histogramas das variaveis quantitativas em dados ORIGINAIS
# (nenhum escalonamento/transformacao aplicado). Um .png por variavel.
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Releitura limpa do dataset: o `df` das secoes anteriores ja carrega colunas
# derivadas (limite, excede, temp_delta, grupo_modo) e dtypes ajustados, entao
# nao serve como fonte dos dados originais.
df_hist = pd.read_csv('../data/dataset_pre_processado.csv')

FIG_DIR = '../assets/graficos'
os.makedirs(FIG_DIR, exist_ok=True)


def bins_discretos(serie):
    """Bins de largura fixa que separam todos os valores distintos da serie.

    Colunas discretas (poucos valores repetidos muitas vezes) precisam de bordas
    alinhadas aos dados: com bins automaticos, valores vizinhos caem no mesmo
    intervalo e deixam de aparecer como barras separadas. A largura usada e o
    menor intervalo entre valores consecutivos observados.
    """
    unicos = np.sort(serie.unique())
    passo = np.diff(unicos).min() if len(unicos) > 1 else 1
    return np.arange(unicos[0] - passo / 2, unicos[-1] + passo, passo)


# Variaveis listadas em "Variaveis analisadas" na subsecao de normalidade da
# documentacao (issues #39 e #44, ambas no singular: a variavel quantitativa).
# (coluna, rotulo do eixo X, bins); 'discretos' aplica a regra da funcao acima.
VARIAVEIS_HIST = [
    ('temp', 'Temperatura (°C)', 40),
]

for coluna, rotulo_x, bins in VARIAVEIS_HIST:
    serie = pd.to_numeric(df_hist[coluna]).dropna()

    if isinstance(bins, str):
        bins = bins_discretos(serie)

    # Com muitos bins a barra fica estreita e a borda branca a encobre.
    n_bins = bins if isinstance(bins, int) else len(bins) - 1
    largura_borda = 0.8 if n_bins <= 80 else 0

    plt.figure(figsize=(8, 5))
    plt.hist(serie, bins=bins, color='#4C72B0',
             edgecolor='white', linewidth=largura_borda)
    plt.title(f'Histograma de {coluna} — dados originais (n = {len(serie)})')
    plt.xlabel(rotulo_x)
    plt.ylabel('Frequência')

    caminho = f'{FIG_DIR}/hist_{coluna}.png'
    plt.savefig(caminho, dpi=150, bbox_inches='tight')
    plt.close()

    print(f'{coluna}: n={len(serie)}, unicos={serie.nunique()}, bins={n_bins}, '
          f'min={serie.min()}, max={serie.max()} -> {caminho}')

## 5. Histogramas dos dados escalonados — Min-Max por `key_area`

Item 3d do artefato. Aplica a equação de normalização definida no item 3c da documentação (Min-Max dentro de cada `key_area`, com as constantes do item 3b) e gera o histograma da variável escalonada, para comparação com a Seção 4.

$$temp\_scaled_{i} = \frac{temp_{i} - \min(temp_{k})}{\max(temp_{k}) - \min(temp_{k})}$$

A transformação é feita em memória; o `data/dataset_pre_processado.csv` não é sobrescrito.

In [ ]:
# Item 3d: histograma de temp APOS o escalonamento Min-Max por key_area,
# conforme a equacao do item 3c da documentacao.
import os

import matplotlib.pyplot as plt
import pandas as pd

df_esc = pd.read_csv('../data/dataset_pre_processado.csv')

FIG_DIR = '../assets/graficos'
os.makedirs(FIG_DIR, exist_ok=True)

# Min-Max dentro de cada key_area: as constantes min/max sao as do item 3b.
minimo = df_esc.groupby('key_area')['temp'].transform('min')
maximo = df_esc.groupby('key_area')['temp'].transform('max')
df_esc['temp_scaled'] = (df_esc['temp'] - minimo) / (maximo - minimo)

serie = df_esc['temp_scaled'].dropna()

plt.figure(figsize=(8, 5))
plt.hist(serie, bins=40, color='#4C72B0', edgecolor='white', linewidth=0.8)
plt.title(f'Histograma de temp escalonada \u2014 Min-Max por key_area (n = {len(serie)})')
plt.xlabel('Temperatura normalizada por key_area (adimensional, [0, 1])')
plt.ylabel('Frequ\u00eancia')

caminho = f'{FIG_DIR}/hist_temp_scaled.png'
plt.savefig(caminho, dpi=150, bbox_inches='tight')
plt.close()
print('figura salva em:', caminho)

# Conferencia 1: cada key_area deve ficar exatamente no intervalo [0, 1].
print()
print('min/max de temp_scaled por key_area (esperado: 0 e 1):')
print(df_esc.groupby('key_area')['temp_scaled'].agg(['min', 'max']).round(6).to_string())

# Conferencia 2: efeito da transformacao sobre a forma da distribuicao.
print()
print(f'{"":<22}{"original":>12}{"escalonada":>12}')
for rotulo, metodo in [('assimetria', 'skew'), ('curtose (excesso)', 'kurt'),
                       ('media', 'mean'), ('mediana', 'median')]:
    a = getattr(df_esc['temp'], metodo)()
    b = getattr(df_esc['temp_scaled'], metodo)()
    print(f'{rotulo:<22}{a:>12.4f}{b:>12.4f}')

n_orig = len(df_esc['temp'].dropna())
correl = df_esc['temp'].corr(df_esc['temp_scaled'])
print(f'{"n":<22}{n_orig:>12}{len(serie):>12}')
print(f'{"correl. c/ original":<22}{"":>12}{correl:>12.6f}')

## 6. Histogramas por `key_area` — antes e depois do escalonamento

Um arquivo por região, cada um com dois painéis lado a lado: `temp` em °C (antes) e `temp_scaled` em [0, 1] (depois do Min-Max daquela região).

O objetivo é verificar o efeito do escalonamento na unidade em que ele foi definido. Como dentro de uma mesma `key_area` o Min-Max é uma **única** transformação linear, a forma da distribuição não se altera: os dois painéis de cada figura têm o mesmo desenho, mudando apenas a escala do eixo X. A assimetria e a curtose impressas em cada painel confirmam numericamente essa invariância.

In [ ]:
# Histogramas por key_area: antes (°C) e depois (Min-Max da regiao, [0, 1]).
# Um .png por regiao, para inspecionar o escalonamento na unidade em que ele
# foi definido -- cada modelo do projeto e treinado por key_area (item 3a).
import os
import re

import matplotlib.pyplot as plt
import pandas as pd

df_area = pd.read_csv('../data/dataset_pre_processado.csv')

FIG_DIR = '../assets/graficos'
os.makedirs(FIG_DIR, exist_ok=True)

# Min-Max dentro de cada key_area (equacao do item 3c).
minimo = df_area.groupby('key_area')['temp'].transform('min')
maximo = df_area.groupby('key_area')['temp'].transform('max')
df_area['temp_scaled'] = (df_area['temp'] - minimo) / (maximo - minimo)

BINS = 30


def slug(texto):
    """Nome de arquivo seguro a partir do rotulo da regiao."""
    return re.sub(r'[^a-z0-9]+', '_', texto.strip().lower()).strip('_')


for area, g in df_area.groupby('key_area'):
    orig = g['temp'].dropna()
    esc = g['temp_scaled'].dropna()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    ax1.hist(orig, bins=BINS, color='#4C72B0', edgecolor='white', linewidth=0.8)
    ax1.set_title('Antes do escalonamento')
    ax1.set_xlabel('Temperatura (°C)')
    ax1.set_ylabel('Frequência')

    ax2.hist(esc, bins=BINS, color='#DD8452', edgecolor='white', linewidth=0.8)
    ax2.set_title('Depois do escalonamento (Min-Max da regiao)')
    ax2.set_xlabel('Temperatura normalizada (adimensional, [0, 1])')
    ax2.set_ylabel('Frequência')

    # Assimetria/curtose em cada painel: devem ser identicas nos dois lados.
    for ax, serie in ((ax1, orig), (ax2, esc)):
        texto = (f'assimetria = {serie.skew():+.4f}' + chr(10)
                 + f'curtose = {serie.kurt():+.4f}')
        ax.annotate(texto,
                    xy=(0.97, 0.95), xycoords='axes fraction',
                    ha='right', va='top', fontsize=9,
                    bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.9))

    fig.suptitle(f'{area} — n = {len(orig)}', fontsize=13)
    fig.tight_layout()

    caminho = f'{FIG_DIR}/hist_area_{slug(area)}.png'
    fig.savefig(caminho, dpi=150, bbox_inches='tight')
    plt.close(fig)

    print(f'{area:<16} n={len(orig):>5}  '
          f'skew {orig.skew():+.4f} -> {esc.skew():+.4f}   '
          f'curt {orig.kurt():+.4f} -> {esc.kurt():+.4f}   {caminho}')